# EN -> HR translation

This notebook does the actual translation, using `translator.py` (placed in the same directory). It runs in three phases:

1. **Setup and sanity checks** (cells 1–3). Verify the API key, the translator module, and that a single test string round-trips correctly.
2. **50-record sample test** (cells 4–6). Translate a small balanced sample from each dataset, save it to disk, and pause for manual quality inspection. **Do not skip this gate.** Distortion-marker preservation (especially Should Statements, All-or-Nothing, Catastrophizing) needs to be checked before spending the full budget.
3. **Full runs** (cells 7–10). One cell per dataset block. Each is idempotent: re-running picks up where it left off and never re-translates a cached string.

Cache layout (kept under `processed/translation_cache/`):
- `tqa_patient_question.json`
- `pr_thought.json`
- `pr_reframes.json`        ← reframe_1, reframe_2, reframe_3 share one cache (same target language, same column type)
- `cr_situation.json`
- `cr_thought.json`
- `cr_reframe.json`

Output bilingual CSVs (written by cell 11):
- `processed/therapistqa_bilingual_{train,val,test}.csv`
- `processed/patternreframe_bilingual_{train,val,test}.csv`
- `processed/cognitive_reframing_bilingual.csv`

Expected wall-clock for the full job: ~60–90 minutes including network overhead. Expected spend against the trial credit: ~\$110.

## 1. Imports and configuration

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

# Make translator.py importable. Adjust if the file lives elsewhere.
sys.path.insert(0, str(Path.cwd()))
import translator

PROCESSED_DIR = Path("./processed")
CACHE_DIR = PROCESSED_DIR / "translation_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# TherapistQA
TQA_FILES = {
    "train": PROCESSED_DIR / "therapistqa_train.csv",
    "val":   PROCESSED_DIR / "therapistqa_val.csv",
    "test":  PROCESSED_DIR / "therapistqa_test.csv",
}

# PatternReframe
PR_FILES = {
    "train": PROCESSED_DIR / "patternreframe_train.csv",
    "val":   PROCESSED_DIR / "patternreframe_val.csv",
    "test":  PROCESSED_DIR / "patternreframe_test.csv",
}

# Cognitive Reframing
CR_FILE = PROCESSED_DIR / "cognitive_reframing.csv"

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. API key check

The API key is read from `GOOGLE_TRANSLATE_API_KEY`.

In [5]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get("GOOGLE_TRANSLATE_API_KEY")
if not api_key:
    print("KEY MISSING — set GOOGLE_TRANSLATE_API_KEY in the shell and restart Jupyter.")
else:
    print(f"KEY OK")

KEY OK


## 3. Single-string round-trip

Translate one string and print the result. Confirms that the API is reachable, the key is valid, the project has Translation API enabled, and billing is wired up. Costs <\$0.001.

In [6]:
test_cache = CACHE_DIR / "__roundtrip_test.json"
result = translator.translate_strings(
    strings=["I should be a better partner."],
    cache_path=test_cache,
    verbose=False,
)
for src, tgt in result.items():
    print(f"EN: {src}")
    print(f"HR: {tgt}")
# Clean up — this test cache is not used by the real job.
test_cache.unlink(missing_ok=True)

EN: I should be a better partner.
HR: Trebao bih biti bolji partner.


## 4. Sample test: 50 records per dataset for manual inspection

Pull a small sample stratified across distortion categories where possible. Save the sample to disk for inspection, then **stop** and read it before launching the full job. This is the only opportunity to catch a systematic translation problem (e.g. "should" being mistranslated, losing the distortion marker) before committing the full budget.

Sample size: 50 from TQA, 50 from PR (5 per category × 10 categories), 50 from CR. Total ~7,500 characters, ~\$0.15 of credit.

In [7]:
import random
random.seed(42)

PR_CATEGORIES = [
    "marked_catastrophizing", "marked_discounting_positive", "marked_labeling",
    "marked_mental_filtering", "marked_mind_reading", "marked_fortune_telling",
    "marked_overgeneralization", "marked_personalization",
    "marked_black_and_white", "marked_should_statements",
]

def sample_tqa(n=50):
    df = pd.read_csv(TQA_FILES["train"])
    return df.sample(min(n, len(df)), random_state=42)["patient_question"].tolist()

def sample_pr(per_category=5):
    df = pd.read_csv(PR_FILES["train"])
    samples = []
    for cat in PR_CATEGORIES:
        sub = df[df[cat] > 0]
        if len(sub) == 0:
            continue
        samples.extend(sub.sample(min(per_category, len(sub)), random_state=42)["thought"].tolist())
    return samples

def sample_cr(n=50):
    df = pd.read_csv(CR_FILE)
    sub = df.sample(min(n, len(df)), random_state=42)
    rows = []
    for _, r in sub.iterrows():
        rows.append(("situation", r["situation"]))
        rows.append(("thought", r["thought"]))
        rows.append(("reframe", r["reframe"]))
    return rows

tqa_sample = sample_tqa(50)
pr_sample = sample_pr(5)
cr_sample = sample_cr(20)   # 20 records × 3 columns = 60 strings

print(f"TQA sample: {len(tqa_sample)} questions")
print(f"PR  sample: {len(pr_sample)} thoughts (5 per category, 10 categories)")
print(f"CR  sample: {len(cr_sample)} (situation/thought/reframe) entries")

TQA sample: 50 questions
PR  sample: 50 thoughts (5 per category, 10 categories)
CR  sample: 60 (situation/thought/reframe) entries


In [8]:
# Translate samples. Uses a dedicated sample-cache, separate from the full-job caches.
SAMPLE_CACHE = CACHE_DIR / "__sample_test.json"

all_sample_strings = (
    list(tqa_sample)
    + list(pr_sample)
    + [text for _, text in cr_sample]
)

print("Translating sample…")
sample_map = translator.translate_strings(
    strings=all_sample_strings,
    cache_path=SAMPLE_CACHE,
    batch_size=50,
)
print(f"\nTranslated {len(sample_map):,} unique strings.")

Translating sample…
  cache: 0 entries already present
  inputs: 160 unique non-empty strings
  to translate: 160 strings (0 cache hits)
  total characters to send: 55,228 (~$1.10 at $20/1M)
  done in 2s; cache now has 160 entries

Translated 160 unique strings.


## 5. Inspect sample translations

Cell prints all 150 EN→HR pairs grouped by dataset, with a focus on the PatternReframe sample because it is per-category and lets distortion markers be checked directly. **Read this output.** Look for:

- **Should Statements** translations should contain `trebao`, `trebala`, `treba`, `morao`, `morala`, or `mora`. If they come out as plain indicatives (e.g. "going to"), the distortion marker is lost.
- **All-or-Nothing** translations should preserve `nikad`, `uvijek`, `svi`, `nitko`, `sve`, `ništa`. If softened ("often" instead of "always"), the marker is lost.
- **Catastrophizing** should keep superlatives and disaster vocabulary (`najgore`, `strašno`, `užasno`, `katastrofa`).
- **Mind Reading** should keep verbs of belief about another's thoughts (`mora da misli`, `siguran/sigurna sam da`).
- For Cognitive Reframing, also check that `situation` translations preserve first-person voice and don't shift tense unexpectedly.

Save the sample translations to disk so they can be reviewed offline or shared with an annotator. If anything looks wrong, **stop** and discuss before running cells 7–10.

In [9]:
sample_rows = []

for src in tqa_sample:
    sample_rows.append({"dataset": "TQA", "category": "", "en": src, "hr": sample_map.get(src)})

df_pr_train = pd.read_csv(PR_FILES["train"])
for src in pr_sample:
    # Find the row to discover which category this thought belongs to.
    matches = df_pr_train[df_pr_train["thought"] == src]
    if len(matches):
        row = matches.iloc[0]
        cats = [c.replace("marked_", "") for c in PR_CATEGORIES if row.get(c, 0) > 0]
        cat_label = ", ".join(cats)
    else:
        cat_label = ""
    sample_rows.append({"dataset": "PR", "category": cat_label, "en": src, "hr": sample_map.get(src)})

for col, src in cr_sample:
    sample_rows.append({"dataset": f"CR-{col}", "category": "", "en": src, "hr": sample_map.get(src)})

sample_df = pd.DataFrame(sample_rows)
sample_df.to_csv(PROCESSED_DIR / "translation_audit" / "sample_translations.csv", index=False)

# Print PR sample for category-by-category inspection.
pd.set_option("display.max_colwidth", None)
print("\nPR sample (grouped by category, inspect markers):")
print(sample_df[sample_df["dataset"] == "PR"][["category", "en", "hr"]].to_string(index=False))


PR sample (grouped by category, inspect markers):
                                                                                                               category                                                                                                                                                                                                               en                                                                                                                                                                                                                                 hr
                                                                      catastrophizing, fortune_telling, black_and_white                                                                                                                                                 I regret becoming a vegetarian. I will soon die of malnutrition.                                                                 

## 6. Sample-quality go/no-go

## 7. Full run — TherapistQA (`patient_question`, all 3 splits)

Expected: ~2,526 strings, ~2.2M characters, ~3 minutes, ~\$44 of credit.

In [10]:
tqa_cache = CACHE_DIR / "tqa_patient_question.json"

tqa_inputs = []
for split, path in TQA_FILES.items():
    df = pd.read_csv(path)
    tqa_inputs.extend(df["patient_question"].dropna().astype(str).tolist())

print(f"TQA: {len(tqa_inputs):,} strings across 3 splits")
tqa_map = translator.translate_strings(
    strings=tqa_inputs,
    cache_path=tqa_cache,
    batch_size=50,
)
print(f"TQA done. Cache has {len(tqa_map):,} entries.")

TQA: 2,526 strings across 3 splits
  cache: 0 entries already present
  inputs: 2,526 unique non-empty strings
  to translate: 2,526 strings (0 cache hits)
  total characters to send: 2,198,364 (~$43.97 at $20/1M)
  batch   10/51  (500/2,526)  elapsed 8s  eta 32s
  batch   20/51  (1,000/2,526)  elapsed 16s  eta 24s
  batch   30/51  (1,500/2,526)  elapsed 24s  eta 16s
  batch   40/51  (2,000/2,526)  elapsed 32s  eta 8s
  batch   50/51  (2,500/2,526)  elapsed 40s  eta 0s
  done in 41s; cache now has 2,526 entries
TQA done. Cache has 2,526 entries.


## 8. Full run — PatternReframe `thought` (all 3 splits)

Expected: ~9,688 thoughts, ~860k characters, ~5 minutes, ~\$17 of credit.

In [11]:
pr_thought_cache = CACHE_DIR / "pr_thought.json"

pr_thoughts = []
for split, path in PR_FILES.items():
    df = pd.read_csv(path)
    pr_thoughts.extend(df["thought"].dropna().astype(str).tolist())

print(f"PR thoughts: {len(pr_thoughts):,} strings across 3 splits")
pr_thought_map = translator.translate_strings(
    strings=pr_thoughts,
    cache_path=pr_thought_cache,
    batch_size=50,
)
print(f"PR thoughts done. Cache has {len(pr_thought_map):,} entries.")

PR thoughts: 9,688 strings across 3 splits
  cache: 0 entries already present
  inputs: 9,682 unique non-empty strings
  to translate: 9,682 strings (0 cache hits)
  total characters to send: 863,256 (~$17.27 at $20/1M)
  batch   10/194  (500/9,682)  elapsed 3s  eta 56s
  batch   20/194  (1,000/9,682)  elapsed 6s  eta 50s
  batch   30/194  (1,500/9,682)  elapsed 9s  eta 47s
  batch   40/194  (2,000/9,682)  elapsed 11s  eta 44s
  batch   50/194  (2,500/9,682)  elapsed 14s  eta 41s
  batch   60/194  (3,000/9,682)  elapsed 17s  eta 38s
  batch   70/194  (3,500/9,682)  elapsed 20s  eta 35s
  batch   80/194  (4,000/9,682)  elapsed 23s  eta 32s
  batch   90/194  (4,500/9,682)  elapsed 26s  eta 30s
  batch  100/194  (5,000/9,682)  elapsed 29s  eta 27s
  batch  110/194  (5,500/9,682)  elapsed 32s  eta 24s
  batch  120/194  (6,000/9,682)  elapsed 35s  eta 21s
  batch  130/194  (6,500/9,682)  elapsed 38s  eta 19s
  batch  140/194  (7,000/9,682)  elapsed 41s  eta 16s
  batch  150/194  (7,500/9,68

## 9. Full run — PatternReframe `reframe_1`, `reframe_2`, `reframe_3` (all 3 splits)

All three reframe slots share one cache because they are the same language and the same kind of text, and there is a small but non-zero chance of cross-slot duplication. Translating them together also means resuming the job after a crash is cleaner.

Expected: ~26,500 reframes, ~2.8M characters, ~25 minutes, ~\$56 of credit.

In [12]:
pr_reframes_cache = CACHE_DIR / "pr_reframes.json"

pr_reframes = []
for split, path in PR_FILES.items():
    df = pd.read_csv(path)
    for col in ["reframe_1", "reframe_2", "reframe_3"]:
        pr_reframes.extend(df[col].dropna().astype(str).tolist())

print(f"PR reframes: {len(pr_reframes):,} strings across 3 splits × 3 columns")
pr_reframes_map = translator.translate_strings(
    strings=pr_reframes,
    cache_path=pr_reframes_cache,
    batch_size=50,
)
print(f"PR reframes done. Cache has {len(pr_reframes_map):,} entries.")

PR reframes: 26,507 strings across 3 splits × 3 columns
  cache: 0 entries already present
  inputs: 26,506 unique non-empty strings
  to translate: 26,506 strings (0 cache hits)
  total characters to send: 2,781,540 (~$55.63 at $20/1M)
  batch   10/531  (500/26,506)  elapsed 3s  eta 160s
  batch   20/531  (1,000/26,506)  elapsed 6s  eta 157s
  batch   30/531  (1,500/26,506)  elapsed 9s  eta 152s
  batch   40/531  (2,000/26,506)  elapsed 12s  eta 150s
  batch   50/531  (2,500/26,506)  elapsed 15s  eta 148s
  batch   60/531  (3,000/26,506)  elapsed 18s  eta 145s
  batch   70/531  (3,500/26,506)  elapsed 21s  eta 140s
  batch   80/531  (4,000/26,506)  elapsed 24s  eta 136s
  batch   90/531  (4,500/26,506)  elapsed 28s  eta 135s
  batch  100/531  (5,000/26,506)  elapsed 31s  eta 133s
  batch  110/531  (5,500/26,506)  elapsed 35s  eta 133s
  batch  120/531  (6,000/26,506)  elapsed 38s  eta 130s
  batch  130/531  (6,500/26,506)  elapsed 41s  eta 127s
  batch  140/531  (7,000/26,506)  elapse

## 10. Full run — Cognitive Reframing (`situation`, `thought`, `reframe`)

Each column gets its own cache because `situation` and `thought` heavily duplicate (286 unique thoughts in 600 rows), and translating each only once gives a guaranteed-consistent result across the duplicate rows.

Expected: ~152k characters total, ~30 seconds, ~\$3 of credit.

In [13]:
df_cr = pd.read_csv(CR_FILE)

cr_situation_map = translator.translate_strings(
    strings=df_cr["situation"].dropna().astype(str).tolist(),
    cache_path=CACHE_DIR / "cr_situation.json",
)
cr_thought_map = translator.translate_strings(
    strings=df_cr["thought"].dropna().astype(str).tolist(),
    cache_path=CACHE_DIR / "cr_thought.json",
)
cr_reframe_map = translator.translate_strings(
    strings=df_cr["reframe"].dropna().astype(str).tolist(),
    cache_path=CACHE_DIR / "cr_reframe.json",
)
print("CR done.")

  cache: 0 entries already present
  inputs: 296 unique non-empty strings
  to translate: 296 strings (0 cache hits)
  total characters to send: 29,174 (~$0.58 at $20/1M)
  done in 2s; cache now has 296 entries
  cache: 0 entries already present
  inputs: 286 unique non-empty strings
  to translate: 286 strings (0 cache hits)
  total characters to send: 11,762 (~$0.24 at $20/1M)
  done in 1s; cache now has 286 entries
  cache: 0 entries already present
  inputs: 600 unique non-empty strings
  to translate: 600 strings (0 cache hits)
  total characters to send: 68,876 (~$1.38 at $20/1M)
  batch   10/12  (500/600)  elapsed 3s  eta 1s
  done in 3s; cache now has 600 entries
CR done.


## 11. Build bilingual CSVs

For each dataset, take the original English CSV and add `*_hr` columns for every translated column. The original English columns stay untouched so that any downstream analysis (cross-lingual comparison in Chapter 4.4, reframing evaluation in Chapter 5) can reference both languages from a single file. Row order and row count are preserved exactly.

In [14]:
# TherapistQA
for split, path in TQA_FILES.items():
    df = pd.read_csv(path)
    translator.add_hr_column(df, "patient_question", "patient_question_hr", tqa_map)
    out = PROCESSED_DIR / f"therapistqa_bilingual_{split}.csv"
    df.to_csv(out, index=False)
    nonnull = df["patient_question_hr"].notna().sum()
    print(f"  {out.name}: {len(df):,} rows, {nonnull:,} translated")

# PatternReframe
for split, path in PR_FILES.items():
    df = pd.read_csv(path)
    translator.add_hr_column(df, "thought", "thought_hr", pr_thought_map)
    for col in ["reframe_1", "reframe_2", "reframe_3"]:
        translator.add_hr_column(df, col, f"{col}_hr", pr_reframes_map)
    out = PROCESSED_DIR / f"patternreframe_bilingual_{split}.csv"
    df.to_csv(out, index=False)
    nonnull = df["thought_hr"].notna().sum()
    print(f"  {out.name}: {len(df):,} rows, {nonnull:,} thoughts translated")

# Cognitive Reframing
df_cr_out = pd.read_csv(CR_FILE)
translator.add_hr_column(df_cr_out, "situation", "situation_hr", cr_situation_map)
translator.add_hr_column(df_cr_out, "thought", "thought_hr", cr_thought_map)
translator.add_hr_column(df_cr_out, "reframe", "reframe_hr", cr_reframe_map)
out = PROCESSED_DIR / "cognitive_reframing_bilingual.csv"
df_cr_out.to_csv(out, index=False)
print(f"  {out.name}: {len(df_cr_out):,} rows")

  therapistqa_bilingual_train.csv: 1,768 rows, 1,768 translated
  therapistqa_bilingual_val.csv: 379 rows, 379 translated
  therapistqa_bilingual_test.csv: 379 rows, 379 translated
  patternreframe_bilingual_train.csv: 1,920 rows, 1,920 thoughts translated
  patternreframe_bilingual_val.csv: 961 rows, 961 thoughts translated
  patternreframe_bilingual_test.csv: 6,807 rows, 6,807 thoughts translated
  cognitive_reframing_bilingual.csv: 600 rows


## 12. Final summary

Print a final report card that goes into the thesis methodology section.

In [16]:
total_translated = (
    len(tqa_map) + len(pr_thought_map) + len(pr_reframes_map)
    + len(cr_situation_map) + len(cr_thought_map) + len(cr_reframe_map)
)
total_chars = sum(len(s) for s in (
    list(tqa_map) + list(pr_thought_map) + list(pr_reframes_map)
    + list(cr_situation_map) + list(cr_thought_map) + list(cr_reframe_map)
))

print("=" * 60)
print("Translation summary")
print("=" * 60)
print(f"TQA patient_question:    {len(tqa_map):>7,} unique strings translated")
print(f"PR thought:              {len(pr_thought_map):>7,} unique strings translated")
print(f"PR reframe_1/2/3:        {len(pr_reframes_map):>7,} unique strings translated")
print(f"CR situation:            {len(cr_situation_map):>7,} unique strings translated")
print(f"CR thought:              {len(cr_thought_map):>7,} unique strings translated")
print(f"CR reframe:              {len(cr_reframe_map):>7,} unique strings translated")
print("-" * 60)
print(f"TOTAL unique strings:    {total_translated:>7,}")
print(f"TOTAL EN characters:     {total_chars:>9,}  ({total_chars/1e6:.2f}M)")
print(f"Approx cost:             ${total_chars / 1e6 * 20:.2f}")
print()
print("Bilingual CSVs written under processed/.")
print("Next: Phase F — back-translation BLEU/chrF + distortion-marker audit.")

Translation summary
TQA patient_question:      2,526 unique strings translated
PR thought:                9,682 unique strings translated
PR reframe_1/2/3:         26,506 unique strings translated
CR situation:                296 unique strings translated
CR thought:                  286 unique strings translated
CR reframe:                  600 unique strings translated
------------------------------------------------------------
TOTAL unique strings:     39,896
TOTAL EN characters:     5,952,972  (5.95M)
Approx cost:             $119.06

Bilingual CSVs written under processed/.
Next: Phase F — back-translation BLEU/chrF + distortion-marker audit.
